####Ex 1→ Setup - create the sales dataset(run this first)
Creates a 20-row sales transactions DataFrame with month, city, category, product, sales_rep, qty, revenue, and returned columns. Run this first — all exercises use this df.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Sales transactions — 20 rows across 3 categories, 4 cities, 3 months
schema = StructType([
    StructField("txn_id",   IntegerType(), True),
    StructField("month",    StringType(),  True),
    StructField("city",     StringType(),  True),
    StructField("category", StringType(),  True),
    StructField("product",  StringType(),  True),
    StructField("sales_rep",StringType(),  True),
    StructField("qty",      IntegerType(), True),
    StructField("revenue",  DoubleType(),  True),
    StructField("returned", StringType(),  True),
])

data = [
    (1,  "Jan","Delhi",    "Electronics","Laptop",    "Alice",  2, 150000.0,"No"),
    (2,  "Jan","Mumbai",   "Electronics","Phone",     "Bob",    5,  125000.0,"No"),
    (3,  "Jan","Delhi",    "Clothing",   "Jeans",     "Alice",  8,   19992.0,"No"),
    (4,  "Jan","Bangalore","Grocery",    "Rice",      "Charlie",20,   9000.0,"No"),
    (5,  "Jan","Mumbai",   "Clothing",   "T-Shirt",   "Bob",    15,  14985.0,"Yes"),
    (6,  "Jan","Chennai",  "Electronics","Headphones","Diana",  10,  35000.0,"No"),
    (7,  "Feb","Delhi",    "Electronics","Laptop",    "Alice",  3, 225000.0,"No"),
    (8,  "Feb","Mumbai",   "Grocery",    "Rice",      "Bob",    30,  13500.0,"No"),
    (9,  "Feb","Bangalore","Electronics","Phone",     "Charlie", 4, 100000.0,"Yes"),
    (10, "Feb","Delhi",    "Clothing",   "Jeans",     "Alice",  6,  14994.0,"No"),
    (11, "Feb","Chennai",  "Grocery",    "Oil",       "Diana",  12,  10800.0,"No"),
    (12, "Feb","Mumbai",   "Electronics","Tablet",    "Bob",    2,   90000.0,"No"),
    (13, "Mar","Delhi",    "Grocery",    "Rice",      "Alice",  25,  11250.0,"No"),
    (14, "Mar","Mumbai",   "Electronics","Laptop",    "Bob",    1,   75000.0,"No"),
    (15, "Mar","Bangalore","Clothing",   "T-Shirt",   "Charlie",20,  19980.0,"No"),
    (16, "Mar","Chennai",  "Electronics","Phone",     "Diana",  3,   75000.0,"No"),
    (17, "Mar","Delhi",    "Electronics","Headphones","Alice",  7,   24500.0,"No"),
    (18, "Mar","Mumbai",   "Clothing",   "Jeans",     "Bob",    4,    9996.0,"Yes"),
    (19, "Mar","Bangalore","Grocery",    "Oil",       "Charlie",8,    7200.0,"No"),
    (20, "Mar","Chennai",  "Electronics","Tablet",    "Diana",  1,   45000.0,"No"),
]

df = spark.createDataFrame(data, schema)
spark.conf.set("spark.sql.shuffle.partitions", 4)
df.display()

txn_id,month,city,category,product,sales_rep,qty,revenue,returned
1,Jan,Delhi,Electronics,Laptop,Alice,2,150000.0,No
2,Jan,Mumbai,Electronics,Phone,Bob,5,125000.0,No
3,Jan,Delhi,Clothing,Jeans,Alice,8,19992.0,No
4,Jan,Bangalore,Grocery,Rice,Charlie,20,9000.0,No
5,Jan,Mumbai,Clothing,T-Shirt,Bob,15,14985.0,Yes
6,Jan,Chennai,Electronics,Headphones,Diana,10,35000.0,No
7,Feb,Delhi,Electronics,Laptop,Alice,3,225000.0,No
8,Feb,Mumbai,Grocery,Rice,Bob,30,13500.0,No
9,Feb,Bangalore,Electronics,Phone,Charlie,4,100000.0,Yes
10,Feb,Delhi,Clothing,Jeans,Alice,6,14994.0,No


####Ex 2→ Whole-DataFrame aggregations - no groupBy
Global aggregations across the entire DataFrame — no grouping. These are your first sanity checks on any new dataset. Know how to combine multiple aggregations in a single agg() call.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

# SINGLE AGGREGATION
print(f"Total Rows: {df.count()}")
print(f"Total revenue: {df.agg(sum("revenue")).collect()[0][0]}")
print(f"average renvenue: {df.agg(avg("revenue")).collect()[0][0]}")

# MULTIPLE AGGREGATION  
df.agg(
    count("*")                                   .alias("total_rows"),
    countDistinct("city")                        .alias("unique_cities"),
    countDistinct("sales_rep")                   .alias("unique_reps"),
    round(sum("revenue"), 2)                     .alias("total_revenue"),
    round(avg("revenue"), 2)                     .alias("average_revenue"),
    round(min("revenue"), 2)                     .alias("min_revenue"),
    round(max("revenue"), 2)                     .alias("max_revenue"),
    round(stddev("revenue"), 2)                  .alias("stddev_revenue"),
    sum("qty")                                   .alias("total_qty"),
    count(when(col("returned") == "Yes", 1))     .alias("returned_orders"),
).display()

# Conditional count — count rows matching a condition
from pyspark.sql.functions import when
df.agg(
    count("*")                                            .alias("total"),
    count(when(col("category") == "Electronics", 1))      .alias("electronics_txns"),
    count(when(col("revenue") > 50000, 1))                .alias("high_value_txns"),
    round(
        count(when(col("returned")=="Yes",1)) /
        count("*") * 100, 1)                             .alias("return_rate_pct"),
).display()

Total Rows: 20
Total revenue: 1076197.0
average renvenue: 53809.85


total_rows,unique_cities,unique_reps,total_revenue,average_revenue,min_revenue,max_revenue,stddev_revenue,total_qty,returned_orders
20,4,4,1076197.0,53809.85,7200.0,225000.0,58914.17,186,3


total,electronics_txns,high_value_txns,return_rate_pct
20,10,7,15.0


####Ex 3→ groupBy + agg - the core aggregation pattern
groupBy is used in every single DE pipeline. Master combining multiple aggregation functions in one agg() call — this is how you build summary tables, reports, and Gold layer tables.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

# SINGLE COLUMN groupBy
print("Revenue by category: ")
df.groupBy("category") \
  .agg(round(sum("revenue"), 2).alias("total_revenue")) \
  .orderBy(col("total_revenue").desc()) \
  .display()

# MULTI COLUMN groupBy
print("City + Catgory breakdown")
df.groupBy("city","category") \
  .agg(
      round(sum("revenue"), 2)           .alias("total_revenue"),
      round(avg("revenue"), 2)           .alias("average_revenue"),
      sum("qty")                         .alias("total_qty"),
      count("txn_id")                    .alias("num_transactions"),
      countDistinct("product")           .alias("unique_products"),
  ).orderBy("city","category").display()

# GroupBy with filter before and after
print("=== Electronics only — by city ===")
df.filter(col("category") == "Electronics") \
  .groupBy("city") \
  .agg(
      sum("revenue")      .alias("electronics_revenue"),
      count("*")          .alias("txn_count"),
  ).orderBy(col("electronics_revenue").desc()).display()

# GroupBy then filter (HAVING equivalent)
print("=== Cities with total revenue > 200000 ===")
df.groupBy("city") \
  .agg(round(sum("revenue"),2).alias("total")) \
  .filter(col("total") > 200000) \
  .orderBy(col("total").desc()) \
  .display()

Revenue by category: 


category,total_revenue
Electronics,944500.0
Clothing,79947.0
Grocery,51750.0


City + Catgory breakdown


city,category,total_revenue,average_revenue,total_qty,num_transactions,unique_products
Bangalore,Clothing,19980.0,19980.0,20,1,1
Bangalore,Electronics,100000.0,100000.0,4,1,1
Bangalore,Grocery,16200.0,8100.0,28,2,2
Chennai,Electronics,155000.0,51666.67,14,3,3
Chennai,Grocery,10800.0,10800.0,12,1,1
Delhi,Clothing,34986.0,17493.0,14,2,1
Delhi,Electronics,399500.0,133166.67,12,3,2
Delhi,Grocery,11250.0,11250.0,25,1,1
Mumbai,Clothing,24981.0,12490.5,19,2,2
Mumbai,Electronics,290000.0,96666.67,8,3,3


=== Electronics only — by city ===


city,electronics_revenue,txn_count
Delhi,399500.0,3
Mumbai,290000.0,3
Chennai,155000.0,3
Bangalore,100000.0,1


=== Cities with total revenue > 200000 ===


city,total
Delhi,445736.0
Mumbai,328481.0


####Ex 4→ countDistinct and approx_count_distinct
Counting distinct values is a very common requirement but also one of the most expensive aggregations — it requires all values to be collected and deduplicated. For large datasets, approx_count_distinct gives a fast approximate answer.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Exact distinct counts — expensive on large data
df.agg(
    countDistinct("city")     .alias("unique_cities"),
    countDistinct("category") .alias("unique_categories"),
    countDistinct("product")  .alias("unique_products"),
    countDistinct("sales_rep").alias("unique_reps"),
).display()

# countDistinct inside groupBy
print("=== Unique products per category ===")
df.groupBy("category") \
    .agg(
      countDistinct("product")  .alias("unique_products"),
      countDistinct("city")     .alias("cities_sold_in"),
      count("*")                .alias("total_txns"),
  ).orderBy("category").display()

# approx_count_distinct — ~5% error but much faster on large data
# rsd = relative standard deviation (lower = more accurate but slower)
df.agg(
    countDistinct("product")                    .alias("exact_count"),
    approx_count_distinct("product", rsd=0.05)  .alias("approx_5pct_error"),
    approx_count_distinct("product", rsd=0.01)  .alias("approx_1pct_error"),
).display()

# Practical: unique customers per category per month
# (simulating cust_id using sales_rep as proxy)
df.groupBy("month","category") \
    .agg(
      countDistinct("sales_rep")    .alias("active_reps"),
      countDistinct("city")         .alias("cities_reached"),
  ).orderBy("month","category").display()

unique_cities,unique_categories,unique_products,unique_reps
4,3,8,4


=== Unique products per category ===


category,unique_products,cities_sold_in,total_txns
Clothing,2,3,5
Electronics,4,4,10
Grocery,2,4,5


exact_count,approx_5pct_error,approx_1pct_error
8,8,8


month,category,active_reps,cities_reached
Feb,Clothing,1,1
Feb,Electronics,3,3
Feb,Grocery,2,2
Jan,Clothing,2,2
Jan,Electronics,3,3
Jan,Grocery,1,1
Mar,Clothing,2,2
Mar,Electronics,3,3
Mar,Grocery,2,2


####Ex 5→ collect_list and collect_set - aggregate into arrays
collect_list and collect_set turn multiple rows into one array per group. Use collect_list when order or duplicates matter, collect_set for unique values. Essential for building product lists, tag aggregations, and activity summaries.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

# collect_list — all products per sales rep (keeps duplicates + order)
print("=== Products sold by each rep (with duplicates) ===")
df.groupBy("sales_rep") \
    .agg(
      collect_list("product")            .alias("all_products"),
      collect_set("product")             .alias("unique_products"),
      collect_set("category")            .alias("categories_covered"),
      collect_set("city")                .alias("cities_worked"),
      size(collect_set("product"))       .alias("distinct_product_count"),
      count("*")                         .alias("total_txns"),
  ).orderBy("sales_rep").display()
    
# collect_list then sort the array
print("=== Sorted product list per city ===")
df.groupBy("city") \
  .agg(sort_array(collect_list("product"))   .alias("products_sorted")) \
  .display()

# Practical: build a customer purchase history (one row per customer)
print("=== Monthly product summary ===")
df.groupBy("month") \
  .agg(
      sort_array(collect_set("product"))   .alias("products_sold"),
      sort_array(collect_set("city"))      .alias("active_cities"),
  ).orderBy("month").display()

=== Products sold by each rep (with duplicates) ===


sales_rep,all_products,unique_products,categories_covered,cities_worked,distinct_product_count,total_txns
Alice,"List(Laptop, Jeans, Laptop, Jeans, Rice, Headphones)","List(Laptop, Jeans, Rice, Headphones)","List(Electronics, Clothing, Grocery)",List(Delhi),4,6
Bob,"List(Phone, T-Shirt, Rice, Tablet, Laptop, Jeans)","List(Phone, T-Shirt, Rice, Tablet, Laptop, Jeans)","List(Electronics, Clothing, Grocery)",List(Mumbai),6,6
Charlie,"List(Rice, Phone, T-Shirt, Oil)","List(Rice, Phone, T-Shirt, Oil)","List(Grocery, Electronics, Clothing)",List(Bangalore),4,4
Diana,"List(Headphones, Oil, Phone, Tablet)","List(Headphones, Oil, Phone, Tablet)","List(Electronics, Grocery)",List(Chennai),4,4


=== Sorted product list per city ===


city,products_sorted
Delhi,"List(Headphones, Jeans, Jeans, Laptop, Laptop, Rice)"
Mumbai,"List(Jeans, Laptop, Phone, Rice, T-Shirt, Tablet)"
Bangalore,"List(Oil, Phone, Rice, T-Shirt)"
Chennai,"List(Headphones, Oil, Phone, Tablet)"


=== Monthly product summary ===


month,products_sold,active_cities
Feb,"List(Jeans, Laptop, Oil, Phone, Rice, Tablet)","List(Bangalore, Chennai, Delhi, Mumbai)"
Jan,"List(Headphones, Jeans, Laptop, Phone, Rice, T-Shirt)","List(Bangalore, Chennai, Delhi, Mumbai)"
Mar,"List(Headphones, Jeans, Laptop, Oil, Phone, Rice, T-Shirt, Tablet)","List(Bangalore, Chennai, Delhi, Mumbai)"
